# Variable-Coefficient Poisson Equation Demo
Solve the PDE:
\begin{align}
-\nabla \cdot k \nabla u &= f \quad \text{in} \; \Omega \\
-\mathbf{n} \cdot k \nabla u &= g \quad \text{on} \; \Gamma_n \\
u &= u_d  \quad \text{on} \; \Gamma_d,
\end{align}
where $k$ is the conductivity tensor (a SPD matrix in $\mathbb{R}^{d \times d}$)

In [ ]:
import variable_coefficient_poisson
import mesh, sparse_matrices, meshing, tri_mesh_viewer
import numpy as np

In [ ]:
import scipy, tensors
def constructK(dim, eulerAngles=None, lambdas=None):
    """
    Construct the flattened upper-triangle entries of a symmetric matrix `Q L Q^t`,
    where `Q` is a rotation described by Euler angles, and L is the diagonal matrix with
    entries `lambdas`.
    """
    if eulerAngles is None: eulerAngles = 0 if dim == 2 else [0, 0, 0]
    if lambdas is None: lambdas = [1] * dim
    Q = scipy.spatial.transform.Rotation.from_euler('z' if dim == 2 else 'xyz', eulerAngles).as_matrix()[0:dim, 0:dim]
    return tensors.SymmetricMatrix(Q @ np.diag(lambdas) @ Q.T).data
def spatiallyConstantK(m, eulerAngles=None, lambdas=None):
    """
    A generic but spatially constant conductivity tensor field.
    """
    k_const = constructK(m.embeddingDimension, eulerAngles, lambdas)
    ks = np.empty((m.numElements(), len(k_const)))
    ks[:, :] = k_const[np.newaxis, :]
    return ks
def randomIsotropicK(m, elementsPerGridcell=None):
    """
    A piecewise constant per-grid-cell isotropic conductivity tensor.
    
    elementsPerGridcell[i] lists the tris/tets associated with gridcell i
    """
    k_const = constructK(m.embeddingDimension)
    ks = np.empty((m.numElements(), len(k_const)))
    ks[:, :] = k_const[np.newaxis, :]
    if elementsPerGridcell is not None:
        ks[elementsPerGridcell, :] *= np.random.uniform(low=0.5, high=2, size=len(elementsPerGridcell))[:, np.newaxis, np.newaxis]
    else:
        ks[:, :] *= np.random.uniform(low=0.5, high=2, size=m.numElements())[:, np.newaxis]
    return ks

In [ ]:
def constructExample(m, f = None, ks = None):
    if ks is None: ks = spatiallyConstantK(m)
    if f is None:
        # Nodal values of the forcing function
        f = np.zeros(m.numNodes()) 
        
    # Find the nodes closest to the bounding box corners to apply Dirichlet conditions
    minCornerNode, maxCornerNode = [np.linalg.norm(m.nodes() - m.bbox[c], axis=1).argmin() for c in range(2)]

    neumannElements = []
    neumannFluxes = []
    dirichletNodes = [minCornerNode, maxCornerNode]
    dirichletValues = [0, 1]
    return variable_coefficient_poisson.construct(m, ks, f, neumannElements, neumannFluxes, dirichletNodes, dirichletValues)
    
def runExample(m, f = None, ks = None):
    vp = constructExample(m, f, ks)
    
    # Solve the linear system
    solver = sparse_matrices.CholeskyFactorizer()
    solver.factorize(vp.A)
    u = solver.solve(vp.b)
    return vp, u

## Test on unstructured meshes

In [ ]:
m = mesh.Mesh('../3rdparty/MeshFEM/misc/examples/meshes/square_hole.off', degree=2)
m = mesh.Mesh('../3rdparty/MeshFEM/misc/examples/meshes/bunny_coarse.msh', degree=1)
m = mesh.Mesh('../3rdparty/MeshFEM/misc/examples/meshes/lilium.msh', degree=1)

In [ ]:
vp, u = runExample(m)

v = tri_mesh_viewer.Viewer(m, wireframe=True, scalarField=u)
v.tetShrinkFactor = 0.5
v.show()

In [ ]:
from matplotlib import pyplot as plt
A_sp = vp.A.toSymmetryMode(vp.A.symmetry_mode.NONE).toSciPy()
plt.spy(A_sp, markersize=2);

## Experiment with structured grids

In [ ]:
def stencil(m, ks=None):
    vp = constructExample(m, ks=ks)
    centerNode = np.argmin(np.linalg.norm(m.nodes() - np.mean(m.bbox, axis=0), axis=1))
    result = np.array(vp.A.toSymmetryMode(vp.A.symmetry_mode.NONE).toSciPy().todense()[:, centerNode])
    return result.reshape((3,) * dim)

In [ ]:
dim = 3
grid = [2] * dim
numCells = np.prod(grid)
m = mesh.Mesh(*meshing.triangulatedGrid(grid, dx=1, triangulationRule=1), degree=1)
vp, u = runExample(m)

In [ ]:
v_grid = tri_mesh_viewer.Viewer(m, wireframe=True, scalarField=u)
v_grid.tetShrinkFactor = 0.25
v_grid.show()

In [ ]:
elementsPerCell = np.arange(m.numElements()).reshape((numCells, -1))

In [ ]:
# Random per-cell isotropic properties
stencil(m, ks=randomIsotropicK(m, elementsPerCell))

In [ ]:
# Random per-tri/tet isotropic properties
stencil(m, ks=randomIsotropicK(m))

# Manipulate stencil using material property sliders

In [ ]:
import ipywidgets

if dim == 2: spatiallyConstantStencil = lambda a0, l0, l1: stencil(m, ks=spatiallyConstantK(m, eulerAngles=a0, lambdas=[l0, l1]))
else:        spatiallyConstantStencil = lambda a0, a1, a2, l0, l1, l2: stencil(m, ks=spatiallyConstantK(m, eulerAngles=[a0, a1, a2], lambdas=[l0, l1, l2]))

sliders = {}
numAngleVars = 1 if dim == 2 else 3
numLambdaVars = dim
for i in range(numAngleVars):  sliders[f'a{i}'] = ipywidgets.FloatSlider(value=0, min=-np.pi, max=np.pi)
for i in range(numLambdaVars): sliders[f'l{i}'] = ipywidgets.FloatSlider(value=1, min=0.01, max=10)

ipywidgets.interact(spatiallyConstantStencil, **sliders);